# DML 第二阶段方法结构对照 · Stage 1

**本次仅编写 notebook，未执行任何 Python、单元测试、smoke test 或模拟。**

本 notebook 自包含 A/B/C/E 的二阶段实现，不需要配套 `.py` 文件。原项目只读；仅使用原 DGP2、n=1000、p=5、5-fold、每场景前 200 个已保存数据集，不训练 HGBR。

- A：原无截距 OLS 与 IF-SE（ddof=1）。
- B：无截距对称 Huber，τ=1.345s。
- C：有截距不对称 Huber，τ₋=2s、τ₊=1.5s。
- E：有截距对称 Huber，τ=1.345s；与 B 的模型结构只差截距。

共同尺度 s 是初步无截距 OLS 残差的 1.4826 MAD。B/C/E 使用各自的 sandwich，不套用 OLS SE。C−E 同时改变截断对称性和强度，不能只归因为非对称性。此次不包括 D。

## 运行方式

选择 **Python (environment2)**：`D:/Anoconda/envs/environment2/python.exe`。
默认 `RUN_COMPARE=False`。确认准备运行后，改为 `True`，保存 notebook，再从头运行单元格。`STAGE` 只能是 `compare`，其他阶段明确报错；不自动串联后续实验。

未来输出到桌面 `DML_Stage2_Results/`：**唯一 CSV `results_comparison.csv` + 四张 PNG**。不输出逐次结果、JSON、NPZ、报告或新 notebook。原始逐次计算结果只在内存中；重跑不覆盖已有 compare，需显式指定新输出目录。

In [ ]:
from pathlib import Path

# 只在你准备运行实验时，手动改成 True；默认不会开始模拟。
RUN_COMPARE = True
STAGE = "compare"
DESKTOP = Path(r"C:\Users\Lenovo\Desktop")
NOTEBOOK_PATH = DESKTOP / "DML_Stage2_Comparison.ipynb"
LEGACY = DESKTOP / "DML_Dukes_full_results" / "n1000_R1000_f76477e6a5dc"
PILOT = DESKTOP / "DML_Stage2_Robust_M_Pilot"
OUTPUT = DESKTOP / "DML_Stage2_Results"

## 选择规则：运行前固定

1. 候选必须有三个场景各 200 个有效结果，A/B/C 与旧 pilot 数值核对通过。
2. 任一重尾场景对 A 的 gain + 1.96×配对 MCSE < 0，视为明显恶化，排除该候选。
3. 主要按 Student-t 与 lognormal 的**平均绝对覆盖率**排序。两者共享 X/treatment 的随机流，因此按共同 rep_id 先合成等权平均，再计算 MCSE，不假设两个场景独立。
4. 与第一名差距 ≤ max(1 个百分点, 1.96×配对差值 MCSE) 为不易区分集合；B 在该集合中且合格时优先 B。
5. 否则在该集合中依次比较最大/平均 tail deterioration、绝对 Bias、绝对 Mean(T)、SE 校准误差和区间长度。
6. 暂定方法若平均重尾区间长度超过 A 的 1.10 倍、且偏差和 SE 校准均无改善，则暂停选择、记录原因，不自动改选。

“明显”“差异小”“区间明显加宽”的数值化阈值是本次预先写定的探索性规则，完整配置写入唯一 CSV；不等于正式非劣效检验、多重比较校正或独立验证。选择只产生一个全场景统一方法；无合格候选时记录 blocked。

工作 sandwich 将估计阈值视为固定，不自动包含全部 nuisance/阈值不确定性，也不证明正交性或渐近有效性。

## 依赖、固定常数与工具函数

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import io
import json
import os
import platform
import sys
import tempfile
import time
import uuid
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

# 不允许本 notebook导入时写本地 __pycache__；第三方库在 main 内按需载入。
sys.dont_write_bytecode = True

N, P, R, FOLDS = 1000, 5, 200, 5
THETA_EVAL = 1.0  # 只在评价阶段使用，不传入估计器或尺度函数。
METHODS = ('A', 'B', 'C', 'E')
SCENES = ('gaussian', 't2.1', 'lognormal2')
TAILS = ('t2.1', 'lognormal2')
ERROR_IDS = {'gaussian': 7, 't2.1': 0, 'lognormal2': 0}
METHOD_LABELS = {'A': 'OLS-DML', 'B': 'Symmetric, no intercept',
                 'C': 'Asymmetric + intercept', 'E': 'Symmetric + intercept'}
COLORS = {'A': '#687386', 'B': '#00868B', 'C': '#D87935', 'E': '#7650A3'}
SCORE_TOL, COND_LIMIT = 1e-9, 1e10
BASE_SEED, FOLD_SEED_OFFSET = 20260826, 500000
CSV_NAME = 'results_comparison.csv'
PNG_NAMES = ('coverage_comparison.png', 'bias_se_comparison.png',
             't_stat_comparison.png', 'tail_deterioration.png')
SCHEMA_VERSION = 'stage2_compare_v1'
SELECT_RULE = {
    'coverage_mcse_multiplier': 1.96,
    'practical_tie_tolerance': 0.01,
    'marked_ci_length_ratio': 1.10,
    'required_valid_per_scenario': 200,
    'fallback_method': 'B',
    'tail_weight_t2.1': 0.5,
    'tail_weight_lognormal2': 0.5,
    'rule_version': 'fixed_before_E_results_v1',
}


def sha(path):
    """只读计算文件哈希，用于输入追踪和提交时的并发修改检测。"""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def require(condition, message):
    """不能用可被 python -O 关闭的 assert 承担输入安全检查。"""
    if not condition:
        raise RuntimeError(message)


def text_json(value):
    """结构化配置写入 CSV 的一个字段；绝不生成 JSON 文件。"""
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def load_dependencies():
    """仅用户手动启动有效阶段后加载计算库，当前编写阶段不会调用。"""
    global np, pd, scipy, minimize, plt, threadpool_limits, tqdm
    try:
        import numpy as np
        import pandas as pd
        import scipy
        from scipy.optimize import minimize
        from threadpoolctl import threadpool_limits
        from tqdm.auto import tqdm
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError('依赖缺失，请使用原 environment2 Python；本 notebook不会自动安装包。'+str(exc)) from exc

## 损失、估计方程与 sandwich

In [ ]:
def validate_residuals(d, y):
    """只接受有限的一维同长残差；不暗中补值、裁剪或重新标准化。"""
    d, y = np.asarray(d, dtype=float), np.asarray(y, dtype=float)
    require(d.ndim == 1 and y.shape == d.shape and len(d) >= 4, '残差维度不一致。')
    require(np.isfinite(d).all() and np.isfinite(y).all(), '残差包含 NaN/Inf。')
    require(np.mean(d*d) > 1e-12, 'treatment 残差退化。')
    return d, y


def fit_original(d, y):
    """逐式保留旧 fit_mean_dml：经验 IF 的方差分母为 n-1。"""
    d, y = validate_residuals(d, y)
    jacobian = np.mean(d**2)
    theta = np.mean(d*y)/jacobian
    residual = y-theta*d
    influence = d*residual/jacobian
    se = np.sqrt(np.var(influence, ddof=1)/len(d))
    require(np.isfinite([theta, se]).all() and se > 0, '原始 OLS 的 SE 非法。')
    return dict(theta_hat=float(theta), se=float(se), intercept=0., has_intercept=False,
                scale=np.nan, tau_minus=np.nan, tau_plus=np.nan, clipping_rate=0.,
                clip_left=0., clip_right=0., converged=True, optimizer_success=True,
                optimizer_message='原始 OLS 闭式解', score_norm=float(abs(np.mean(d*residual))),
                score_theta=float(np.mean(d*residual)), score_intercept=np.nan,
                bread_condition=1., near_knots=0)


def ols_mad(d, y):
    """所有稳健方法同用无截距 OLS 初始残差，不能用真实 θ 或 Oracle 误差。"""
    theta = np.mean(d*y)/np.mean(d*d)
    r = y-theta*d
    scale = float(1.4826*np.median(abs(r-np.median(r))))
    require(np.isfinite(scale) and scale > 1e-12, 'MAD 为零或非有限值；不替换尺度。')
    return scale


def huber_score(r, tm, tp):
    """C 的非对称 score；tm==tp 时退化为 B/E 的对称 score。"""
    return np.clip(r, -tm, tp)


def huber_loss(r, tm, tp):
    """score 的连续凸积分，与旧 pilot 的分段二次/线性损失相同。"""
    return np.where(r < -tm, -tm*r-.5*tm**2,
                    np.where(r > tp, tp*r-.5*tp**2, .5*r*r))


def huber_derivative(r, tm, tp):
    """截断结点导数按旧代码取 0，并另记 near_knots。"""
    return ((r > -tm) & (r < tp)).astype(float)


def objective_gradient(beta, z, y, tm, tp):
    r = y-z@beta
    return float(np.mean(huber_loss(r, tm, tp))), -z.T@huber_score(r, tm, tp)/len(y)


def bread_matrix(beta, z, y, tm, tp):
    w = huber_derivative(y-z@beta, tm, tp)
    return z.T@(w[:, None]*z)/len(y)


def sandwich(z, r, tm, tp):
    """工作 sandwich：中心化 meat 分母为 n；带截距必须保留整个 2×2 矩阵。"""
    n = len(r)
    w = huber_derivative(r, tm, tp)
    A = z.T@(w[:, None]*z)/n
    eigmin = float(np.linalg.eigvalsh(A).min())
    condition = float(np.linalg.cond(A))
    require(eigmin > 1e-12 and np.isfinite(condition) and condition <= COND_LIMIT,
            f'bread 奇异或病态：min_eigen={eigmin}, condition={condition}')
    u = z*huber_score(r, tm, tp)[:, None]
    centered = u-u.mean(axis=0)
    meat = centered.T@centered/n
    inv = np.linalg.solve(A, np.eye(A.shape[0]))
    covariance = inv@meat@inv.T/n
    require(np.isfinite(covariance).all() and covariance[-1, -1] > 0, 'sandwich 方差非法。')
    return covariance, u, condition


def fit_robust(d, y, method):
    """沿用旧 pilot 的 BFGS + Newton 精修；E 仅在 B 的设计矩阵中加入常数。"""
    d, y = validate_residuals(d, y)
    s = ols_mad(d, y)
    require(method in ('B', 'C', 'E'), '未知稳健方法。')
    tm, tp = (2.*s, 1.5*s) if method == 'C' else (1.345*s, 1.345*s)
    intercept = method in ('C', 'E')
    z = np.column_stack([np.ones(len(d)), d]) if intercept else d[:, None]
    require(np.linalg.matrix_rank(z) == z.shape[1], '设计矩阵秩不足。')
    initial = np.linalg.lstsq(z, y, rcond=None)[0]
    fun = lambda b: objective_gradient(b, z, y, tm, tp)
    opt = minimize(fun, initial, jac=True, method='BFGS', options={'gtol': 1e-10, 'maxiter': 1000})
    beta = opt.x.copy()
    # 与旧 pilot 相同的精修条件；不改变损失、阈值或初始尺度。
    for _ in range(30):
        value, grad = fun(beta)
        if np.max(abs(grad)) <= 1e-11:
            break
        A = bread_matrix(beta, z, y, tm, tp)
        if np.linalg.cond(A) > COND_LIMIT:
            break
        direction = np.linalg.solve(A, grad)
        accepted = False
        for k in range(35):
            candidate = beta-direction*(.5**k)
            nv, ng = fun(candidate)
            if nv <= value+1e-14*max(1., abs(value)) and np.linalg.norm(ng) < np.linalg.norm(grad):
                beta, accepted = candidate, True
                break
        if not accepted:
            break
    _, grad = fun(beta)
    error = float(np.max(abs(grad)))
    require(np.isfinite(beta).all() and np.isfinite(error) and error <= SCORE_TOL,
            f'估计方程未收敛：error={error}, optimizer={opt.message}')
    r = y-z@beta
    covariance, u, condition = sandwich(z, r, tm, tp)
    return dict(theta_hat=float(beta[-1]), se=float(np.sqrt(covariance[-1, -1])),
                intercept=float(beta[0]) if intercept else 0., has_intercept=intercept,
                scale=s, tau_minus=tm, tau_plus=tp, converged=True,
                optimizer_success=bool(opt.success), optimizer_message=str(opt.message),
                score_norm=error, score_theta=float(u[:, -1].mean()),
                score_intercept=float(u[:, 0].mean()) if intercept else np.nan,
                clipping_rate=float(np.mean((r < -tm) | (r > tp))),
                clip_left=float(np.mean(r < -tm)), clip_right=float(np.mean(r > tp)),
                bread_condition=condition,
                near_knots=int(np.sum(np.isclose(r, -tm, atol=1e-10, rtol=0) |
                                      np.isclose(r, tp, atol=1e-10, rtol=0))))

## 旧数据读取与逐组 A/B/C/E 对照

In [ ]:
def read_sources(args):
    """仅读取已存在的配置、数组清单和旧结果；缺失时不尝试生成或补训练。"""
    legacy, pilot = args.legacy.resolve(), args.pilot.resolve()
    needed = [legacy/'config.json', pilot/'input_manifest.json', pilot/'replication_results.csv',
              pilot/'delivery_manifest.json', pilot/'run_config.json']
    for p in needed:
        require(p.is_file(), f'缺少必需原文件：{p}。请指定正确 --legacy/--pilot；不会生成替代数据。')
    cfg = json.loads(needed[0].read_text(encoding='utf-8'))
    require((cfg['n'], cfg['p'], cfg['n_folds'], cfg['dgp'], cfg['theta0']) ==
            (N, P, FOLDS, 'DGP2_nonlinear', THETA_EVAL), '原配置不符合固定 Stage 1。')
    require(cfg['base_seed'] == BASE_SEED and cfg['fold_seed_offset'] == FOLD_SEED_OFFSET,
            '原种子配置发生变化。')
    require(tuple(x['name'] for x in cfg['scenarios']) == SCENES, '原场景顺序或标签变化。')
    oldconfig = json.loads((pilot/'run_config.json').read_text(encoding='utf-8'))
    require(sha(legacy/'config.json') == oldconfig['source']['legacy_config_sha256'], '原配置哈希变化。')
    require(oldconfig['planned_replications'] == R and oldconfig['settings'] ==
            dict(b_kappa=1.345, c_minus=2., c_plus=1.5, d_kappa=1.), '旧 pilot 配置不同。')
    for name, actual in [('numpy', np.__version__), ('scipy', scipy.__version__)]:
        require(actual == cfg['environment'][name],
                f'{name} 版本 {actual} 与旧实验 {cfg["environment"][name]} 不同；请使用原 environment2。')
    manifest_list = json.loads((pilot/'input_manifest.json').read_text(encoding='utf-8'))
    manifest = {(x['scenario'], int(x['rep_id'])): x for x in manifest_list}
    expected = {(s, r) for s in SCENES for r in range(R)}
    require(len(manifest_list) == 600 and set(manifest) == expected, '旧 pilot 输入清单不是完整 600 组。')
    delivery = json.loads((pilot/'delivery_manifest.json').read_text(encoding='utf-8'))
    hashes = [h for p, h in delivery['files'].items() if Path(p).name == 'replication_results.csv']
    require(len(hashes) == 1 and sha(pilot/'replication_results.csv') == hashes[0], '旧逐次结果哈希不一致。')
    old = pd.read_csv(pilot/'replication_results.csv')
    old = old[old.method.isin(('A', 'B', 'C'))].copy()
    require(len(old) == 1800 and not old.duplicated(['scenario', 'rep_id', 'method']).any(), '旧 A/B/C 结果缺失或重复。')
    require(old.status.eq('ok').all(), '旧 A/B/C 有失败，不能直接作为数值对照。')
    old = old.set_index(['scenario', 'rep_id', 'method'])
    for scene, rep in expected:
        stem = f'{scene}_{rep:04d}'
        require((legacy/'tasks'/(stem+'.json')).is_file() and (legacy/'arrays'/(stem+'.npz')).is_file(),
                f'缺少已保存数据 {stem}；本阶段禁止训练 HGBR。')
    return cfg, manifest, old, {str(p): sha(p) for p in needed}


def load_saved(args, manifest, scene, rep):
    """正确映射：g_hat=treatment，ell_hat=outcome；完全不读取旧 m_hat。"""
    legacy = args.legacy.resolve()
    stem = f'{scene}_{rep:04d}'
    taskpath = legacy/'tasks'/(stem+'.json')
    ap = legacy/'arrays'/(stem+'.npz')
    source = manifest[(scene, rep)]
    taskhash, arrayhash = sha(taskpath), sha(ap)
    require(taskhash == source['task_sha256'] and arrayhash == source['array_sha256'], '输入哈希变化：'+stem)
    task = json.loads(taskpath.read_text(encoding='utf-8'))
    require(arrayhash == task['arrays_sha256'], '原 task/NPZ 不匹配：'+stem)
    original = next(row for row in task['rows'] if row['method'] == 'original_mean_dml')
    seed = BASE_SEED+100000+1000*ERROR_IDS[scene]+rep
    require(original['dataset_seed'] == source['seed'] == seed and
            original['crossfit_seed'] == source['crossfit_seed'] == seed+FOLD_SEED_OFFSET,
            '种子不一致：'+stem)
    with np.load(ap, allow_pickle=False) as saved:
        required = ('g_hat', 'ell_hat', 'd_res', 'y_res', 'fold_ids')
        require(all(k in saved.files for k in required), '缺少 A 的 nuisance/residual 数组：'+stem)
        a = {k: saved[k].copy() for k in required}
    for k, value in a.items():
        require(value.shape == (N,) and np.isfinite(value).all(), f'{stem}/{k} 维度或有限性异常。')
    folds = a['fold_ids']
    require(np.issubdtype(folds.dtype, np.integer), 'fold 标签类型异常。')
    labels, counts = np.unique(folds, return_counts=True)
    require(np.array_equal(labels, np.arange(FOLDS)) and np.all(counts == N//FOLDS), 'fold 标签/大小异常。')
    require(hashlib.sha256(folds.tobytes()).hexdigest() == source['fold_hash'] == original['fold_hash'], 'fold 哈希不一致。')
    for key, target in [('g_hat', 'm_prediction_hash'), ('ell_hat', 'ell_prediction_hash')]:
        require(hashlib.sha256(a[key].tobytes()).hexdigest() == source[target], 'nuisance 预测哈希不一致。')
    # 直接使用已核验的旧 A 残差，不重建 DGP，不训练，不重算 folds。
    return a['d_res'], a['y_res'], original, [(str(taskpath), taskhash), (str(ap), arrayhash)]


def compare_one(d, y, scene, rep, old, original):
    """一次 replication 四个方法共享同一输入；失败 coverage 留空。"""
    rows = []
    for method in METHODS:
        row = dict(scenario=scene, rep_id=rep, method=method, status='failed',
                   theta_hat=np.nan, se=np.nan, T=np.nan, covered=np.nan, ci_length=np.nan,
                   converged=False, optimizer_success=False, error='', warnings='[]',
                   legacy_match=False if method in ('A', 'B', 'C') else None,
                   legacy_maxdiff=np.nan, near_knots=0, scale=np.nan, tau_minus=np.nan, tau_plus=np.nan, clipping_rate=np.nan, clip_left=np.nan, clip_right=np.nan, intercept=np.nan, score_norm=np.nan, bread_condition=np.nan)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            try:
                fit = fit_original(d, y) if method == 'A' else fit_robust(d, y, method)
                row.update(fit)
                t = (fit['theta_hat']-THETA_EVAL)/fit['se']
                require(np.isfinite(t), '标准化统计量非有限值。')
                row.update(status='ok', T=float(t), covered=float(abs(t) <= 1.96), ci_length=3.92*fit['se'])
            except Exception as exc:
                row.update(status='failed', covered=np.nan, T=np.nan, error=type(exc).__name__+': '+str(exc))
        row['warnings'] = text_json([str(w.message) for w in caught])
        if method in ('A', 'B', 'C') and row['status'] == 'ok':
            previous = old.loc[(scene, rep, method)]
            keys = ['theta_hat', 'se', 'intercept', 'clipping_rate']
            if method != 'A': keys += ['scale', 'tau_minus', 'tau_plus']
            differences = [abs(float(row[k])-float(previous[k])) for k in keys]
            if method == 'A':
                differences += [abs(row['theta_hat']-original['theta_hat']), abs(row['se']-original['theta_se'])]
            row['legacy_maxdiff'] = max(differences)
            row['legacy_match'] = bool(row['legacy_maxdiff'] <= 1e-10)
            # 不把不匹配“修正”成看起来一致；保存诊断并禁止方法选择。
            if not row['legacy_match']:
                row['error'] = '与旧 pilot 数值不一致；禁止生成有效选择记录。'
        rows.append(row)
    return rows

## 汇总、配对比较、选择与异常记录

In [ ]:
def mcse(values):
    """配对差值的 Monte Carlo SE，分母用实际有效配对数。"""
    x = np.asarray(values, dtype=float)
    return float(np.std(x, ddof=1)/np.sqrt(len(x))) if len(x) > 1 else np.nan


def coverage_matrix(raw, scene):
    """失败仍为 NaN，绝不当作未覆盖；pivot 也检测重复 ID。"""
    return raw[raw.scenario.eq(scene)].pivot(index='rep_id', columns='method', values='covered').reindex(range(R))


def tail_matrix(raw):
    """t 与 lognormal 共用 X/treatment 流：先按 rep_id 等权合成再计算方差。"""
    return (coverage_matrix(raw, TAILS[0])+coverage_matrix(raw, TAILS[1]))/2


def summarize(raw):
    """唯一 CSV 中的 summary 行：所有指标按有效结果，失败数同时披露。"""
    records = []
    for scene in SCENES:
        for method in METHODS:
            g = raw[raw.scenario.eq(scene) & raw.method.eq(method)]
            v = g[g.status.eq('ok')]
            row = dict(record_type='summary', scenario=scene, method=method,
                       completed=len(g), valid=len(v), failed=len(g)-len(v), coverage=np.nan, coverage_mcse=np.nan, bias=np.nan, empirical_sd=np.nan, mean_se=np.nan, mean_T=np.nan, sd_T=np.nan, reject_left=np.nan, reject_right=np.nan,
                       convergence_rate=float(g.converged.mean()),
                       optimizer_flag_false=int((~g.optimizer_success).sum()),
                       warning_records=int(g.warnings.ne('[]').sum()),
                       nonfinite_statistics=int((~np.isfinite(g[['theta_hat','se','T']])).any(axis=1).sum()),
                       legacy_mismatch_count=int((g.legacy_match == False).sum()) if method != 'E' else 0,
                       legacy_maxdiff=g.legacy_maxdiff.max())
            if len(v):
                cov = float(v.covered.mean())
                row.update(bias=float((v.theta_hat-THETA_EVAL).mean()), empirical_sd=v.theta_hat.std(ddof=1),
                           mean_se=v.se.mean(), theta_rmse=float(np.sqrt(np.mean((v.theta_hat-THETA_EVAL)**2))),
                           coverage=cov, coverage_mcse=np.sqrt(cov*(1-cov)/len(v)),
                           mean_T=v['T'].mean(), sd_T=v['T'].std(ddof=1),
                           reject_left=float((v['T'] < -1.96).mean()), reject_right=float((v['T'] > 1.96).mean()),
                           mean_ci_length=v.ci_length.mean(), mean_scale=v.scale.mean(),
                           mean_tau_minus=v.tau_minus.mean(), mean_tau_plus=v.tau_plus.mean(),
                           mean_clipping_rate=v.clipping_rate.mean(), mean_clip_left=v.clip_left.mean(),
                           mean_clip_right=v.clip_right.mean(),
                           mean_intercept=v.intercept.mean() if method in ('C','E') else np.nan,
                           se_calibration=v.se.mean()/v.theta_hat.std(ddof=1),
                           max_equation_error=v.score_norm.max(), max_bread_condition=v.bread_condition.max(),
                           near_knot_count=int(v.near_knots.sum()))
            records.append(row)
    summary = pd.DataFrame(records)
    for i, row in summary.iterrows():
        matrix = coverage_matrix(raw, row.scenario)[[row.method, 'A']] if row.method != 'A' else None
        diff = (matrix.iloc[:, 0]-matrix.iloc[:, 1]).dropna() if matrix is not None else pd.Series(np.zeros(int(row.valid)))
        summary.loc[i, 'paired_coverage_gain_vs_A'] = diff.mean()
        summary.loc[i, 'paired_gain_mcse'] = mcse(diff)
        summary.loc[i, 'paired_valid'] = len(diff)
        gauss = summary[summary.scenario.eq('gaussian') & summary.method.eq(row.method)].iloc[0]
        summary.loc[i, 'delta_tail'] = 0. if row.scenario == 'gaussian' else gauss.get('coverage',np.nan)-row.get('coverage',np.nan)
        summary.loc[i, 'delta_tail_mcse'] = 0. if row.scenario == 'gaussian' else np.hypot(gauss.get('coverage_mcse',np.nan), row.get('coverage_mcse',np.nan))
    return summary


def paired_comparisons(raw):
    """差值统一为 method-reference；同时保留六种指定结构/基准对照。"""
    records = []
    pairs = [('E','B'), ('C','E'), ('C','B'), ('B','A'), ('C','A'), ('E','A')]
    matrices = {s: coverage_matrix(raw,s) for s in SCENES}
    matrices['heavy_average'] = tail_matrix(raw)
    for method, reference in pairs:
        for scene, matrix in matrices.items():
            diff = (matrix[method]-matrix[reference]).dropna()
            error = mcse(diff)
            row = dict(record_type='paired_comparison', scenario=scene, method=method, reference=reference,
                       paired_valid=len(diff), coverage_difference=diff.mean(), paired_mcse=error,
                       difference_low=diff.mean()-1.96*error, difference_high=diff.mean()+1.96*error,
                       clearly_different=bool(len(diff)>1 and abs(diff.mean())>max(.01,1.96*error)))
            if scene in SCENES:
                fields=['theta_hat','se','T','ci_length','clipping_rate']
                left=raw[raw.scenario.eq(scene)&raw.method.eq(method)&raw.status.eq('ok')].set_index('rep_id')
                right=raw[raw.scenario.eq(scene)&raw.method.eq(reference)&raw.status.eq('ok')].set_index('rep_id')
                joint=left[fields].join(right[fields],lsuffix='_m',rsuffix='_r',how='inner')
                for field in fields:
                    values=joint[field+'_m']-joint[field+'_r']
                    row[field+'_difference']=values.mean()
                    row[field+'_difference_mcse']=mcse(values)
                row['empirical_sd_difference']=left.theta_hat.std(ddof=1)-right.theta_hat.std(ddof=1)
            if scene != 'gaussian':
                gd=(matrices['gaussian'][method]-matrices['gaussian'][reference]).dropna()
                row['deterioration_difference']=gd.mean()-diff.mean()
                row['deterioration_difference_mcse']=np.hypot(mcse(gd),error)
            row['interpretation']=('t/lognormal 按共同 rep_id 合成，保留相关性' if scene=='heavy_average'
                                   else '差值=method-reference；C-E 同时改变对称性和截断强度')
            records.append(row)
    return records


def choose_method(raw, summary):
    """只输出一个全场景统一选择；不完整或审计失败时生成 blocked 记录。"""
    lookup=summary.set_index(['scenario','method'])
    q=tail_matrix(raw)
    audits_ok=bool(raw[raw.method.isin(('A','B','C'))].legacy_match.eq(True).all())
    baseline_complete=all(int(lookup.loc[(s,'A'),'valid'])==R for s in SCENES)
    candidate_rows=[]
    for method in ('B','C','E'):
        valid=all(int(lookup.loc[(s,method),'valid'])==R for s in SCENES)
        tails=[lookup.loc[(s,method)] for s in TAILS]
        bases=[lookup.loc[(s,'A')] for s in TAILS]
        means=q[method].dropna()
        row=dict(record_type='selection_candidate',scenario='all',method=method,
                 selection_rule_version=SELECT_RULE['rule_version'],
                 complete=valid, source_audit_pass=audits_ok,
                 heavy_average_coverage=means.mean(), heavy_average_mcse=mcse(means),
                 heavy_complete_pairs=len(means), gaussian_coverage=lookup.loc[('gaussian',method)].get('coverage',np.nan),
                 mean_tail_deterioration=np.mean([x.get('delta_tail',np.nan) for x in tails]),
                 worst_tail_deterioration=max(x.get('delta_tail',np.nan) for x in tails),
                 mean_abs_bias=np.mean([abs(x.get('bias',np.nan)) for x in tails]),
                 mean_abs_T=np.mean([abs(x.get('mean_T',np.nan)) for x in tails]),
                 mean_se_calibration_error=np.mean([abs(x.get('se_calibration',np.nan)-1) for x in tails]),
                 mean_ci_length=np.mean([x.get('mean_ci_length',np.nan) for x in tails]))
        harm=False
        for s in TAILS:
            c=lookup.loc[(s,method)]
            gain=c.get('paired_coverage_gain_vs_A',np.nan); se=c.get('paired_gain_mcse',np.nan)
            row[s+'_gain_vs_A']=gain;row[s+'_gain_mcse']=se
            row[s+'_clear_harm']=bool(np.isfinite([gain,se]).all() and gain+1.96*se<0)
            harm=harm or row[s+'_clear_harm']
        lengths=np.mean([x.get('mean_ci_length',np.nan) for x in bases])
        row['ci_length_ratio_vs_A']=row['mean_ci_length']/lengths if lengths>0 else np.nan
        basebias=np.mean([abs(x.get('bias',np.nan)) for x in bases])
        basecal=np.mean([abs(x.get('se_calibration',np.nan)-1) for x in bases])
        row['width_only_concern']=bool(row['ci_length_ratio_vs_A']>1.10 and
                                      row['mean_abs_bias']>=basebias and row['mean_se_calibration_error']>=basecal)
        row['eligible']=bool(valid and baseline_complete and audits_ok and not harm)
        row['eligibility_reason']=('完整且未发现显著重尾恶化' if row['eligible'] else
                                   '结果不完整、原方法不匹配、A 不完整或存在重尾恶化；禁止静默填补')
        candidate_rows.append(row)
    eligible=[r for r in candidate_rows if r['eligible']]
    selected='';reason='没有通过资格核验的候选；保留诊断，不自动选择。'
    if eligible:
        top=max(eligible,key=lambda x:x['heavy_average_coverage'])
        indistinguishable=[]
        for item in eligible:
            diff=(q[top['method']]-q[item['method']]).dropna()
            gap=float(diff.mean());error=mcse(diff)
            item.update(raw_coverage_leader=top['method'], gap_to_leader=gap,
                        gap_to_leader_mcse=error,tie_bound=max(.01,1.96*error))
            item['within_tie_set']=bool(gap<=item['tie_bound'])
            if item['within_tie_set']:indistinguishable.append(item)
        fallback=next((r for r in indistinguishable if r['method']=='B'),None)
        chosen=fallback or min(indistinguishable,key=lambda r:(r['worst_tail_deterioration'],
                 r['mean_tail_deterioration'],r['mean_abs_bias'],r['mean_abs_T'],
                 r['mean_se_calibration_error'],r['mean_ci_length'],r['method']))
        if chosen['width_only_concern']:
            reason='暂定候选 '+chosen['method']+' 的改善疑似仅靠扩大区间；暂停选择，需人工判断，不自动改选。'
        else:
            selected=chosen['method']
            reason=('B 与覆盖率领先者的差异小于预设实际/MCSE 阈值，按预设规则保留 B。' if fallback else
                    '先按重尾平均绝对覆盖率构造不可清楚区分集合，再依次按 tail loss、偏差、T、SE 校准和长度选择。')
    for row in candidate_rows:row['selected']=bool(selected and row['method']==selected)
    decision=dict(record_type='selection',scenario='all',method=selected,selected_method=selected,
                  selection_status='complete' if selected else 'blocked',selection_reason=reason,
                  selection_rule=text_json(SELECT_RULE),selection_replications=R,
                  baseline_complete=baseline_complete,source_audit_pass=audits_ok,
                  validation_scope='仅探索性 R=200 选择；不构成独立确认；后续 full 必须使用新的独立随机流',
                  common_method_for_all_distributions=True)
    return candidate_rows+[decision],decision


def anomaly_rows(raw):
    """异常按场景/方法/类型汇总到同一 CSV，不写逐 replication 数据。"""
    result=[]
    for (scene,method),g in raw.groupby(['scenario','method'],sort=False):
        for field in ('error','warnings'):
            counts=Counter(str(x) for x in g[field] if str(x) not in ('','[]','nan'))
            for message,count in counts.items():
                result.append(dict(record_type='anomaly',scenario=scene,method=method,
                                   anomaly_type=field,anomaly_message=message,anomaly_count=count))
        flagged=g[g.status.eq('ok') & ~g.optimizer_success]
        if len(flagged):
            result.append(dict(record_type='anomaly',scenario=scene,method=method,
                               anomaly_type='optimizer_flag_but_equations_passed',anomaly_count=len(flagged),
                               anomaly_message='优化器原始 success=False，但最终方程和 sandwich 通过验收',
                               max_equation_error=flagged.score_norm.max()))
    return result

## 作图、原子保存与 Stage 1 主函数

In [ ]:
def make_plots(summary):
    """四张图先在内存中完成；均用零基线，coverage 不隐藏 0 或 95% 参考线。"""
    images={}
    x=np.arange(len(SCENES));width=.18
    labels=['Gaussian','Student-t(2.1)','Lognormal(2)']
    def finish(fig,name):
        fig.tight_layout(rect=(0,0,1,.93))
        buf=io.BytesIO();fig.savefig(buf,format='png',dpi=200,bbox_inches='tight')
        images[name]=buf.getvalue();plt.close(fig)
    def group_bars(ax,key,title,error_key=None):
        for j,m in enumerate(METHODS):
            v=summary[summary.method.eq(m)].set_index('scenario').reindex(SCENES)
            values=pd.to_numeric(v[key],errors='coerce') if key in v else np.full(3,np.nan)
            xpos=x+(j-1.5)*width
            ax.bar(xpos,values,width,color=COLORS[m],label=m+': '+METHOD_LABELS[m])
            if error_key and error_key in v:
                ax.errorbar(xpos,values,yerr=1.96*v[error_key],fmt='none',ecolor='#333333',capsize=2,lw=.8)
        ax.axhline(0,color='black',lw=.7);ax.set_xticks(x,labels);ax.set_title(title)
        ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
        lo,hi=ax.get_ylim();ax.set_ylim(min(0,lo),max(0,hi))
    fig,ax=plt.subplots(figsize=(10,6))
    group_bars(ax,'coverage','Coverage; error bars = +/-1.96 MCSE','coverage_mcse')
    ax.axhline(.95,color='black',ls='--',label='Nominal 95%');ax.set_ylim(0,1.05)
    ax.legend(fontsize=8,loc='lower left');fig.suptitle('Stage 1 | n=1000, R=200 | A/B/C/E | fixed saved OOF predictions')
    finish(fig,PNG_NAMES[0])
    fig,axes=plt.subplots(1,3,figsize=(15,5))
    for ax,key,title in zip(axes,['bias','empirical_sd','mean_se'],['Bias','Empirical SD','Mean estimated SE']):
        group_bars(ax,key,title)
    handles,legend=axes[0].get_legend_handles_labels();fig.legend(handles,legend,loc='upper center',ncol=4,fontsize=8)
    finish(fig,PNG_NAMES[1])
    fig,axes=plt.subplots(2,2,figsize=(12,9))
    for ax,key,title in zip(axes.flat,['mean_T','sd_T','reject_left','reject_right'],
                             ['Mean(T)','SD(T)','Left rejection rate','Right rejection rate']):
        group_bars(ax,key,title)
        if key=='sd_T':ax.axhline(1,color='black',ls='--',lw=1)
        if key.startswith('reject'):
            ax.axhline(.025,color='black',ls='--',lw=1);ax.set_ylim(0,1)
    handles,legend=axes[0,0].get_legend_handles_labels();fig.legend(handles,legend,loc='upper center',ncol=4,fontsize=8)
    finish(fig,PNG_NAMES[2])
    fig,ax=plt.subplots(figsize=(10,6))
    for j,m in enumerate(METHODS):
        v=summary[summary.method.eq(m)].set_index('scenario').reindex(TAILS)
        xpos=np.arange(2)+(j-1.5)*width
        ax.bar(xpos,v.delta_tail,width,color=COLORS[m],label=m+': '+METHOD_LABELS[m])
        ax.errorbar(xpos,v.delta_tail,yerr=1.96*v.delta_tail_mcse,fmt='none',ecolor='#333333',capsize=2,lw=.8)
    ax.axhline(0,color='black',lw=.7);ax.set_ylim(-1,1)
    ax.set_xticks([0,1],['Gaussian - Student-t','Gaussian - Lognormal'])
    ax.set_ylabel('Coverage difference');ax.set_title('Heavy-tail deterioration; error bars = +/-1.96 MCSE')
    ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True);ax.legend(fontsize=8)
    finish(fig,PNG_NAMES[3])
    return images


def inspect_existing_output(output):
    """不创建目录。已有 compare 不覆盖；避免与未来 full/large 的选择记录脱节。"""
    target=output/CSV_NAME
    if not output.exists():return None,None
    require(output.is_dir(),'输出路径不是目录。')
    unexpected=[p.name for p in output.iterdir() if not (p.name==CSV_NAME or (p.is_file() and p.suffix.lower()=='.png'))]
    require(not unexpected,'输出目录存在不允许的文件/目录，请使用独立空目录：'+str(unexpected))
    if not target.exists():return None,None
    previous=pd.read_csv(target)
    require({'stage','record_type','schema_version'}.issubset(previous.columns),'已有 CSV 的结构无法安全合并。')
    require(not previous.stage.eq('compare').any(),'compare 结果已存在；为保护选择记录拒绝覆盖。需要另跑时请显式指定新的 --output 目录。')
    require(not previous.stage.isin(['full','large']).any(),'已有后续阶段但缺失 compare；选择链不完整，拒绝接入。')
    return previous,sha(target)


def atomic_save_csv(frame,target,previous_hash):
    """在同目录使用临时 CSV + os.replace；失败删除临时文件，旧 CSV 保持完整。"""
    require((sha(target) if target.exists() else None)==previous_hash,'运行期间 CSV 被其他进程修改，拒绝覆盖。')
    fd,temp=tempfile.mkstemp(prefix='.writing_',suffix='.csv',dir=target.parent)
    try:
        with os.fdopen(fd,'w',encoding='utf-8-sig',newline='') as stream:
            frame.to_csv(stream,index=False);stream.flush();os.fsync(stream.fileno())
        # 临时 CSV 只用于原子提交；不是第二个结果文件，正常/异常退出均清理。
        require((sha(target) if target.exists() else None)==previous_hash,'提交前 CSV 被其他进程修改。')
        os.replace(temp,target)
    finally:
        if os.path.exists(temp):os.unlink(temp)


def save_png(data,target):
    """图像也原子替换，临时文件仅有 PNG 后缀，异常时清理。"""
    fd,temp=tempfile.mkstemp(prefix='.writing_',suffix='.png',dir=target.parent)
    try:
        with os.fdopen(fd,'wb') as stream:
            stream.write(data);stream.flush();os.fsync(stream.fileno())
        os.replace(temp,target)
    finally:
        if os.path.exists(temp):os.unlink(temp)


def run_compare(args):
    """唯一实验函数；没有测试/Oracle/后续阶段自动调用。"""
    started=time.perf_counter()
    output=args.output.resolve()
    protected=[args.legacy.resolve(),args.pilot.resolve()]
    require(all(output!=p and p not in output.parents for p in protected),'输出不能位于原项目或原结果目录内。')
    previous,previous_hash=inspect_existing_output(output)
    cfg,manifest,old,source_files=read_sources(args)
    rows=[];used_files=dict(source_files)
    with threadpool_limits(limits=1),tqdm(total=3*R,desc='Stage 1: saved datasets',unit='dataset') as progress:
        for rep in range(R):
            for scene in SCENES:
                d,y,original,files=load_saved(args,manifest,scene,rep)
                used_files.update(files)
                rows.extend(compare_one(d,y,scene,rep,old,original))
                progress.update(1)
            if (rep+1)%20==0:
                progress.write(f'每场景完成 {rep+1}/{R}；仅二阶段 A/B/C/E，没有训练 HGBR。')
    raw=pd.DataFrame(rows)
    require(len(raw)==2400 and not raw.duplicated(['scenario','rep_id','method']).any(),'结果组数异常。')
    # 选中记录提交前再次核验原文件，以免运行期间的外部编辑混入结果。
    for path,digest in used_files.items():require(sha(path)==digest,'原输入在运行中改变：'+path)
    summary=summarize(raw)
    comparisons=paired_comparisons(raw)
    selection,decision=choose_method(raw,summary)
    timestamp=datetime.now().astimezone().isoformat()
    run_id='compare_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    config_records=[]
    config_items={
        'legacy_path':str(args.legacy.resolve()),'pilot_path':str(args.pilot.resolve()),
        'notebook_sha256_on_disk':sha(NOTEBOOK_PATH),'python':platform.python_version(),
        'numpy':np.__version__,'scipy':scipy.__version__,'pandas':pd.__version__,
        'first_stage_parameters':cfg['hgbr_params_seed_zero'],
        'replication_ids':'0..199; three saved scenarios; no new random stream',
        'seed_rule':'20260826+100000+1000*error_id+rep; error_id Gaussian=7,t/LN=0; fold seed +500000',
        'nuisance_mapping':'g_hat=treatment; ell_hat=outcome; do not use legacy m_hat',
        'scale':'1.4826*median(abs(r_OLS-median(r_OLS))); OLS has no intercept',
        'thresholds':{'B':[1.345,1.345],'C':[2.,1.5],'E':[1.345,1.345]},
        'sandwich':'centered meat /n; A original IF ddof=1; threshold treated fixed',
        'optimizer':'original BFGS gtol=1e-10/maxiter=1000 + up to 30 Newton refinements',
        'source_manifest_files_sha256':source_files,'used_input_files_count':len(used_files),'used_input_files_digest':hashlib.sha256(text_json(used_files).encode('utf-8')).hexdigest(),'selection_rule':SELECT_RULE,
        'input_files_unchanged':True,'no_nuisance_training':True,
        'seconds':time.perf_counter()-started,
        'scope':'stage1 only; no D method; full/large unimplemented; no automatic chaining',
        'limitations':'exploratory selection on shared data; working sandwich; no orthogonality/asymptotic guarantee',
        'E_vs_C_caveat':'both symmetry and clipping intensity differ',
    }
    for key,value in config_items.items():
        config_records.append(dict(record_type='config',scenario='all',method='',config_key=key,
                                   config_value=text_json(value) if isinstance(value,(dict,list,bool)) else str(value)))
    new=pd.concat([summary,pd.DataFrame(comparisons+selection+anomaly_rows(raw)+config_records)],ignore_index=True,sort=False)
    for key,value in dict(stage='compare',schema_version=SCHEMA_VERSION,run_id=run_id,
                          n=N,p=P,R=R,folds=FOLDS,theta_evaluation=THETA_EVAL,
                          dgp='DGP2_nonlinear',created_at=timestamp).items():new[key]=value
    # 统一表结构支持未来追加其他 stage；本版不生成虚假的 full/large 记录。
    combined=new if previous is None else pd.concat([previous,new],ignore_index=True,sort=False)
    images=make_plots(summary)
    output.mkdir(parents=True,exist_ok=True)
    # CSV 为权威结果，先原子提交。PNG 若被占用，只报错而不触发重新模拟。
    atomic_save_csv(combined,output/CSV_NAME,previous_hash)
    for name,data in images.items():save_png(data,output/name)
    print('Stage 1 完成，已停止；不会执行 full 或 large。')
    print('CSV:',output/CSV_NAME)
    print('统一选择:',decision['selected_method'] or '未选择 / blocked')
    print('原因:',decision['selection_reason'])
    print('本阶段使用选择数据；不得当作独立确认实验。')
    return 0 if decision['selection_status']=='complete' else 2

## 手动执行 Stage 1：每 20 次显示进度，完成后停止

In [ ]:
# 本格只有在 RUN_COMPARE=True 时才开始；没有自动 smoke/Oracle/后续阶段。
if STAGE != "compare":
    raise ValueError("本次只实现 Stage 1 compare；full/large 未实现，不会自动运行其他阶段。")
if RUN_COMPARE:
    load_dependencies()
    args = argparse.Namespace(stage=STAGE, legacy=LEGACY, pilot=PILOT, output=OUTPUT)
    exit_status = run_compare(args)
    if exit_status != 0:
        print("Stage 1 计算已结束，但未产生可用于后续阶段的有效选择；请查看 CSV 的 selection/anomaly 行。")
else:
    print("运行开关关闭。尚未执行实验、测试、训练或方法选择，也未创建结果目录。")

## 读取已保存的唯一汇总 CSV 和四张图（本格不触发模拟）

In [ ]:
# 仅在你手动执行本格、且结果已经存在时读取结果；不触发任何模拟。
if (OUTPUT / "results_comparison.csv").is_file():
    load_dependencies()
    from IPython.display import display, Image
    results = pd.read_csv(OUTPUT / "results_comparison.csv")
    display(results.loc[(results.stage == "compare") & (results.record_type == "summary")])
    for _, row in results.loc[(results.stage == "compare") & (results.record_type == "selection")].iterrows():
        print("选择状态：", row.get("selection_status"))
        print("统一主方法：", row.get("selected_method"))
        print("选择原因：", row.get("selection_reason"))
    for name in PNG_NAMES:
        image_path = OUTPUT / name
        if image_path.is_file():
            display(Image(filename=str(image_path)))
        else:
            print("图像缺失：", image_path, "；不要为了补图重新启动模拟。")
else:
    print("暂无结果；本格不会自动运行 Stage 1。")